In [ ]:
from music21 import converter
import os
from collections import defaultdict
import pickle

# =========================
# CONFIG
# =========================

CONFIG = {
    "dataset_path": "/Volumes/TOSHIBA  2T/CIPI_dataset/scores",
    "grid": 1/6,
    "allowed_meters": {
        '4/4','2/4','3/4','3/8','6/8',
        '9/8','5/8','5/4','7/8','7/4','12/8'
    }
}

# =========================
# FEATURE EXTRACTION
# =========================

def get_measure_rhythm(measure):
    return tuple(el.duration.quarterLength for el in measure.notesAndRests)

def has_spillover(measure):
    return any(
        n.tie and n.tie.type in ("start", "continue")
        for n in measure.notes
    )

def extract_rhythms(score):
    rhythm_dict = defaultdict(lambda: defaultdict(int))

    if not score.parts:
        return rhythm_dict

    part = score.parts[0]

    ts = part.recurse().getElementsByClass('TimeSignature')
    if not ts:
        return rhythm_dict

    meter = ts[0].ratioString
    measures = list(part.getElementsByClass('Measure'))

    for i, m in enumerate(measures):
        pattern = get_measure_rhythm(m)

        if pattern:
            rhythm_dict[meter][pattern] += 1

        # spillover (tie across measure)
        if has_spillover(m) and i + 1 < len(measures):
            next_m = measures[i + 1]
            combined = pattern + get_measure_rhythm(next_m)
            rhythm_dict[meter][combined] += 1

    return rhythm_dict


def scan_dataset(path):
    global_dict = defaultdict(lambda: defaultdict(int))

    for root, _, files in os.walk(path):
        for file in files:
            if file.startswith("._"):
                continue

            if not file.lower().endswith((".xml", ".musicxml", ".mid", ".midi")):
                continue

            filepath = os.path.join(root, file)

            try:
                score = converter.parse(filepath)
            except Exception as e:
                print(f"Skipping {file}: {e}")
                continue

            local_dict = extract_rhythms(score)

            # merge counts
            for meter, patterns in local_dict.items():
                for pat, count in patterns.items():
                    global_dict[meter][pat] += count

    return global_dict


# =========================
# CLEANING & NORMALIZATION
# =========================

def quantize(d, grid):
    return round(d / grid) * grid

def is_ternary_group(durs, beat=1.0, tol=1e-6):
    return (
        len(durs) == 3 and
        abs(sum(durs) - beat) < tol and
        all(abs(d - beat/3) < tol for d in durs)
    )

def get_beat_length(meter):
    return 1.5 if meter in ['6/8','9/8','12/8'] else 1.0


def clean_rhythm(pattern, grid, beat):
    q = [quantize(float(d), grid) for d in pattern]
    q = [d for d in q if d > 0]

    cleaned = []
    i = 0

    while i < len(q):
        # preserve ternary groups
        if i + 2 < len(q):
            group = q[i:i+3]
            if is_ternary_group(group, beat):
                cleaned.extend(group)
                i += 3
                continue

        d = q[i]

        # merge tiny durations
        if d < beat / 6 and i + 1 < len(q):
            q[i + 1] += d
        else:
            cleaned.append(d)

        i += 1

    return tuple(cleaned)


def clean_all(rhythm_dict, grid):
    cleaned_dict = defaultdict(lambda: defaultdict(int))

    for meter, patterns in rhythm_dict.items():
        beat = get_beat_length(meter)

        for pattern, count in patterns.items():
            cleaned = clean_rhythm(pattern, grid, beat)

            if cleaned:
                cleaned_dict[meter][cleaned] += count

    return cleaned_dict


# =========================
# POSTPROCESSING
# =========================

def approx_equal(x, targets, tol=1e-6):
    return any(abs(x - t) < tol for t in targets)


def filter_patterns(rhythm_dict, allowed_meters):
    final_dict = {}

    for meter, patterns in rhythm_dict.items():

        if meter not in allowed_meters:
            continue

        final_dict[meter] = {}

        for pattern, count in patterns.items():
            pattern = tuple(x for x in pattern if x != 0)

            if not pattern:
                continue

            total = sum(pattern)

            # meter-specific validation
            if meter == '12/8' and not approx_equal(total, [6, 12]):
                continue
            if meter == '9/8' and not approx_equal(total, [4.5, 9]):
                continue

            final_dict[meter][pattern] = count

    return final_dict


# =========================
# PIPELINE
# =========================

def build_rhythm_dataset(config=CONFIG):
    print("Scanning dataset...")
    raw = scan_dataset(config["dataset_path"])

    print("Cleaning rhythms...")
    cleaned = clean_all(raw, config["grid"])

    print("Filtering patterns...")
    final = filter_patterns(cleaned, config["allowed_meters"])

    print("Done.")
    return final


# =========================
# SAVE / LOAD
# =========================

def save_dataset(data, path="rhythm_dataset.pkl"):
    with open(path, "wb") as f:
        pickle.dump(data, f)

def load_dataset(path="rhythm_dataset.pkl"):
    with open(path, "rb") as f:
        return pickle.load(f)